### Unity Catalog Governance: Key Concepts & Examples

* **Catalog → Schema → Table hierarchy**: Catalogs are top-level containers, schemas organize tables/views within catalogs, and tables/views hold the actual data.
  * Example: `main.sales.customers` (catalog: `main`, schema: `sales`, table: `customers`)
  * SQL: `SELECT * FROM demo_catalog.sales.transactions`

* **Access control (GRANT/REVOKE)**: Use SQL commands to grant or revoke privileges (SELECT, INSERT, etc.) at catalog, schema, table, or view level for fine-grained security.
  * Example: `GRANT SELECT ON TABLE demo_catalog.sales.transactions TO `arpandasgupta56@gmail.com``
  * SQL: `REVOKE SELECT ON TABLE demo_catalog.sales.transactions FROM `arpandasgupta56@gmail.com``
  * Grant to group: `GRANT SELECT ON SCHEMA demo_catalog.sales TO `data_analysts``

* **Data lineage**: Unity Catalog tracks how data flows between tables, views, and notebooks, helping with auditing and compliance.
  * Example: You can view lineage in the Databricks UI to see which notebooks or jobs created or modified a table/view.
  * SQL: `DESCRIBE HISTORY demo_catalog.sales.transactions` (shows table change history)

* **Managed vs external tables**: Managed tables store data in Databricks-managed locations; external tables reference data stored outside Databricks (e.g., S3).
  * Managed Example: `CREATE TABLE demo_catalog.sales.transactions (id INT) USING DELTA`
  * External Example: `CREATE TABLE demo_catalog.sales.ext_transactions USING DELTA LOCATION 's3://my-bucket/data/'`

---

**Next steps in this notebook:**
1. Create catalog & schemas
2. Register Delta tables
3. Set up permissions
4. Create views for controlled access

---

**Practical Scenarios & More Examples:**
* Use catalogs to separate business domains (e.g., `finance`, `sales`, `marketing`).
* Grant SELECT on views to analysts, but restrict access to raw tables for compliance.
* Track lineage to audit who changed data and when.
* Use external tables for data stored in S3, managed tables for internal analytics.
* Create a row-level security view:
  * `CREATE OR REPLACE VIEW demo_catalog.sales.us_transactions AS SELECT * FROM demo_catalog.sales.transactions WHERE country = 'US'`
* Grant INSERT privilege:
  * `GRANT INSERT ON TABLE demo_catalog.sales.transactions TO `data_engineers``
* Show table change history:
  * `DESCRIBE HISTORY demo_catalog.sales.transactions`


In [0]:
# Create a catalog and schema for organizing your data assets
# Catalogs are top-level containers; schemas organize tables/views within catalogs
# Run this cell to create a catalog and schema if they don't exist

spark.sql("CREATE CATALOG IF NOT EXISTS demo_catalog")
spark.sql("CREATE SCHEMA IF NOT EXISTS demo_catalog.sales")

In [0]:
# Register a Delta table in your schema
# Delta tables support ACID transactions and time travel
# This example creates a managed Delta table from sample data

spark.sql("""
CREATE TABLE IF NOT EXISTS demo_catalog.sales.transactions
USING DELTA
AS SELECT '2026-01-16' AS event_time, 'purchase' AS event_type, 100 AS amount
""")

In [0]:
# Set up permissions using GRANT statements
# This example grants SELECT privilege on the table to a user or group
# Replace 'account_user' with the actual user or group name

spark.sql("GRANT SELECT ON TABLE demo_catalog.sales.transactions TO `arpandasgupta56@gmail.com`")

In [0]:
# Create a view to provide controlled access to sensitive data
# Views can restrict columns or rows for different users/groups

spark.sql("""
CREATE OR REPLACE VIEW demo_catalog.sales.transactions_view AS
SELECT event_time, event_type FROM demo_catalog.sales.transactions
""")